# Worker memory analysis

This notebook analyzes both the newer AI memory-lab artifacts (`analysis/timeline.csv`) and the older bounded IG stream artifacts (`memory.csv`). For the AI lab it plots cgroup, RSS, PSS, managed, private-dirty, LOH, chart-render, and AI-review evidence.

The attribution is diagnostic rather than proof of ownership: cgroup/PSS/private-dirty growth can include runtime, native libraries, HTTP, SQLite, charting, and allocator behavior.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('ggplot')

# Set this to a run directory to analyze a different run.
RUN_NAME = 'worker-ai-memory-20260718T143200Z'
RUN_DIRECTORY_OVERRIDE = None

candidates = []
if RUN_DIRECTORY_OVERRIDE:
    candidates.append(Path(RUN_DIRECTORY_OVERRIDE))
candidates.extend([
    Path('../artifacts/worker-ai-memory-lab') / RUN_NAME,
    Path('artifacts/worker-ai-memory-lab') / RUN_NAME,
    Path.cwd() / 'artifacts' / 'worker-ai-memory-lab' / RUN_NAME,
    Path('../artifacts') / RUN_NAME,
    Path('artifacts') / RUN_NAME,
    Path.cwd() / 'artifacts' / RUN_NAME,
])
RUN_DIRECTORY = next((p.resolve() for p in candidates if p.exists()), None)
if RUN_DIRECTORY is None:
    raise FileNotFoundError('Could not find the run directory. Set RUN_DIRECTORY_OVERRIDE.')

NEW_FORMAT = (RUN_DIRECTORY / 'analysis' / 'timeline.csv').exists()
if NEW_FORMAT:
    CSV_PATH = RUN_DIRECTORY / 'analysis' / 'timeline.csv'
    SUMMARY_PATH = RUN_DIRECTORY / 'analysis' / 'summary.json'
else:
    CSV_PATH = RUN_DIRECTORY / 'memory.csv'
    SUMMARY_PATH = RUN_DIRECTORY / 'summary.json'

print(f'Run directory: {RUN_DIRECTORY}')
print(f'Format: {"AI memory lab" if NEW_FORMAT else "legacy live IG stream"}')
print(f'CSV: {CSV_PATH.stat().st_size:,} bytes')

In [ ]:
summary = json.loads(SUMMARY_PATH.read_text()) if SUMMARY_PATH.exists() else {}
raw = pd.read_csv(CSV_PATH)
if NEW_FORMAT:
    raw['timestamp'] = pd.to_datetime(raw['observedAtUtc'], utc=True)
    for column in raw.columns:
        if column not in {'observedAtUtc', 'timestamp', 'activeOperations'}:
            raw[column] = pd.to_numeric(raw[column], errors='coerce')
    worker = raw.sort_values('timestamp').reset_index(drop=True)
    worker['managedMiB'] = worker['managedCommittedMiB']
    worker['nativePrivateDirtyMiB'] = worker['privateDirtyMiB']
    worker['sampleIntervalSeconds'] = worker['timestamp'].diff().dt.total_seconds()
else:
    raw['timestamp'] = pd.to_datetime(raw['timestampUtc'], utc=True)
    numeric_columns = ['pid', 'workingSetBytes', 'privateMemoryBytes', 'cpuSeconds', 'databaseBytes', 'walBytes', 'shmBytes']
    raw[numeric_columns] = raw[numeric_columns].apply(pd.to_numeric, errors='coerce')
    raw['rssMiB'] = raw['workingSetBytes'] / 2**20
    raw['privateMiB'] = raw['privateMemoryBytes'] / 2**20
    raw['databaseMiB'] = raw['databaseBytes'] / 2**20
    raw['walMiB'] = raw['walBytes'] / 2**20
    raw['shmMiB'] = raw['shmBytes'] / 2**20
    worker = raw.loc[raw['process'].eq('Trading.Cli')].sort_values('timestamp').reset_index(drop=True)
    worker['managedMiB'] = np.nan
    worker['sampleIntervalSeconds'] = worker['timestamp'].diff().dt.total_seconds()

if len(worker) < 2:
    raise ValueError('Expected at least two memory samples.')
worker['elapsedHours'] = (worker['timestamp'] - worker['timestamp'].iloc[0]).dt.total_seconds() / 3600
worker['elapsedMinutes'] = worker['elapsedHours'] * 60
worker['phase'] = np.where(worker['elapsedHours'] < 1, 'Warm-up', 'Steady state')
warmup = worker.loc[worker['elapsedHours'] < 1]
steady = worker.loc[worker['elapsedHours'] >= 1]

print(f'Samples: {len(worker):,}')
print(f'Measured window: {worker.timestamp.iloc[0]} to {worker.timestamp.iloc[-1]}')
if NEW_FORMAT:
    print(f'Peak cgroup: {worker.cgroupMiB.max():.2f} MiB; peak PSS: {worker.pssMiB.max():.2f} MiB')
    print(f'AI prompt records: {summary.get("ai", {}).get("promptCount", "see prompt-summary.json")}')
else:
    print(f'Legacy Trading.Cli samples: {len(worker):,}')
worker.head(3)

## 1. Headline memory metrics

In [ ]:
def slope_per_hour(frame, column):
    valid = frame[['elapsedHours', column]].dropna()
    if len(valid) < 2 or valid.elapsedHours.nunique() < 2:
        return np.nan
    return float(np.polyfit(valid.elapsedHours, valid[column], 1)[0])

metrics = []
for label, column in [('Cgroup', 'cgroupMiB'), ('RSS', 'rssMiB'), ('PSS', 'pssMiB'), ('Managed committed', 'managedMiB'), ('Private dirty', 'nativePrivateDirtyMiB')]:
    if column in worker and worker[column].notna().any():
        metrics.extend([
            (f'Initial {label} (MiB)', worker[column].dropna().iloc[0]),
            (f'Peak {label} (MiB)', worker[column].max()),
            (f'Final {label} (MiB)', worker[column].dropna().iloc[-1]),
            (f'Steady {label} slope (MiB/hour)', slope_per_hour(steady, column)),
        ])
metrics.extend([
    ('Baseline-to-peak cgroup (MiB)', worker.cgroupMiB.max() - worker.cgroupMiB.iloc[0]) if 'cgroupMiB' in worker else ('Baseline-to-peak RSS (MiB)', worker.rssMiB.max() - worker.rssMiB.iloc[0]),
    ('Average sample interval (s)', worker.sampleIntervalSeconds.dropna().mean()),
    ('Gaps over 15 seconds', int((worker.sampleIntervalSeconds > 15).sum())),
])
pd.DataFrame(metrics, columns=['metric', 'value']).style.format({'value': '{:.3f}'})

## 2. Memory evolution

For the AI lab, this shows cgroup current alongside process RSS/PSS and managed memory. The first hour is marked as warm-up; inspect the later plateau for retention or leak-like growth.

In [ ]:
columns = [("cgroupMiB", "Cgroup current", '#b71c1c'), ("rssMiB", "RSS", '#1565c0'), ("pssMiB", "PSS", '#00838f'), ("managedMiB", "Managed committed", '#2e7d32'), ("nativePrivateDirtyMiB", "Private dirty", '#ef6c00')]
plot_columns = [(column, label, color) for column, label, color in columns if column in worker and worker[column].notna().any()]
plot = worker.set_index('timestamp')[[column for column, _, _ in plot_columns]].resample('1min').median()
fig, ax = plt.subplots(figsize=(15, 7))
for column, label, color in plot_columns:
    ax.plot(plot.index, plot[column], label=label, color=color, linewidth=1.6)
warmup_boundary = worker.timestamp.iloc[0] + pd.Timedelta(hours=1)
ax.axvline(warmup_boundary, color='#555', linestyle='--', linewidth=1.3, label='Warm-up boundary')
peak_column = 'cgroupMiB' if 'cgroupMiB' in worker else 'rssMiB'
peak = worker.loc[worker[peak_column].idxmax()]
ax.scatter([peak.timestamp], [peak[peak_column]], color='#000', zorder=5)
ax.annotate(f'Peak {peak[peak_column]:.1f} MiB', (peak.timestamp, peak[peak_column]), xytext=(10, 10), textcoords='offset points')
ax.set_title('Worker memory over time')
ax.set_ylabel('MiB')
ax.legend(loc='best')
fig.autofmt_xdate()
plt.show()

## 3. Memory composition and operations

In [ ]:
if NEW_FORMAT:
    components = [c for c in ['cgroupMiB', 'cgroupFileMiB', 'privateDirtyMiB', 'managedCommittedMiB', 'lohMiB', 'pohMiB'] if c in worker]
    fig, axes = plt.subplots(2, 1, figsize=(15, 10), sharex=True)
    for column in ['cgroupFileMiB', 'privateDirtyMiB', 'managedCommittedMiB', 'lohMiB']:
        if column in worker:
            axes[0].plot(worker.timestamp, worker[column], label=column, linewidth=1.4)
    axes[0].set_ylabel('MiB')
    axes[0].set_title('Memory components')
    axes[0].legend(loc='best')
    axes[1].plot(worker.timestamp, worker['threadCount'], label='Threads')
    axes[1].plot(worker.timestamp, worker['streamDispatcherDepth'], label='Dispatcher depth')
    axes[1].plot(worker.timestamp, worker['streamIngestorDepth'], label='Ingestor depth')
    axes[1].set_ylabel('Count / depth')
    axes[1].set_title('Thread and stream pressure')
    axes[1].legend(loc='best')
    fig.autofmt_xdate()
    plt.show()
else:
    fig, ax1 = plt.subplots(figsize=(14, 6))
    ax1.plot(worker.timestamp, worker.privateMiB, color='#ef6c00', label='Private memory')
    ax1.plot(worker.timestamp, worker.rssMiB, color='#1565c0', alpha=0.6, label='RSS')
    ax1.set_ylabel('Worker memory (MiB)')
    ax2 = ax1.twinx()
    for column, label, color in [('databaseMiB', 'SQLite file', '#2e7d32'), ('walMiB', 'WAL', '#8e24aa'), ('shmMiB', 'SHM', '#6d4c41')]:
        ax2.step(worker.timestamp, worker[column], where='post', label=label, color=color)
    ax2.set_ylabel('SQLite files (MiB)')
    lines = ax1.get_lines() + ax2.get_lines()
    ax1.legend(lines, [line.get_label() for line in lines], loc='upper left')
    ax1.set_title('SQLite growth versus worker memory')
    fig.autofmt_xdate()
    plt.show()

## 4. AI and chart operation evidence

In [ ]:
if NEW_FORMAT:
    prompt_path = RUN_DIRECTORY / 'analysis' / 'prompt-summary.json'
    checkpoint_path = RUN_DIRECTORY / 'analysis' / 'operation-checkpoints.json'
    prompts = pd.DataFrame(json.loads(prompt_path.read_text())) if prompt_path.exists() else pd.DataFrame()
    checkpoints = pd.DataFrame(json.loads(checkpoint_path.read_text())) if checkpoint_path.exists() else pd.DataFrame()
    if not checkpoints.empty:
        completed = checkpoints.loc[checkpoints['completedAtUtc'].notna()].copy()
        completed['deltaCgroupMiB'] = completed['after'].apply(lambda x: x.get('cgroupMiB') if isinstance(x, dict) else np.nan) - completed['before'].apply(lambda x: x.get('cgroupMiB') if isinstance(x, dict) else np.nan)
        operation_summary = completed.groupby('operation').agg(count=('operation', 'size'), meanDeltaCgroupMiB=('deltaCgroupMiB', 'mean'), maxDeltaCgroupMiB=('deltaCgroupMiB', 'max'), totalPayloadBytes=('payloadBytes', 'sum')).sort_values('meanDeltaCgroupMiB', ascending=False)
        display(operation_summary)
        operation_summary[['meanDeltaCgroupMiB', 'maxDeltaCgroupMiB']].plot(kind='bar', figsize=(12, 5), title='Memory delta at completed operations', ylabel='MiB')
        plt.show()
    if not prompts.empty:
        prompts['requestedAtUtc'] = pd.to_datetime(prompts['requestedAtUtc'], utc=True)
        prompts['completedAtUtc'] = pd.to_datetime(prompts['completedAtUtc'], utc=True)
        prompts['requestedCgroupMiB'] = prompts['nearestRequestedMemory'].apply(lambda x: x.get('cgroupMiB') if isinstance(x, dict) else np.nan)
        prompts['completedCgroupMiB'] = prompts['nearestCompletedMemory'].apply(lambda x: x.get('cgroupMiB') if isinstance(x, dict) else np.nan)
        prompts['deltaCgroupMiB'] = prompts['completedCgroupMiB'] - prompts['requestedCgroupMiB']
        display(prompts[['promptName', 'status', 'requestedAtUtc', 'durationMilliseconds', 'inputTokens', 'outputTokens', 'requestedCgroupMiB', 'completedCgroupMiB', 'deltaCgroupMiB']])
        fig, ax = plt.subplots(figsize=(15, 5))
        ax.plot(worker.timestamp, worker.cgroupMiB, color='#b71c1c', label='Cgroup current')
        for _, row in prompts.iterrows():
            color = '#1565c0' if row['promptName'].startswith('daily') else '#6a1b9a'
            ax.axvspan(row.requestedAtUtc, row.completedAtUtc, color=color, alpha=0.12)
        ax.set_title('Cgroup memory with AI call intervals')
        ax.set_ylabel('MiB')
        ax.legend()
        fig.autofmt_xdate()
        plt.show()
else:
    print('This is a legacy memory.csv run; AI and operation checkpoint files are not available.')

## 5. Interpretation

For the AI lab, compare the post-warm-up cgroup/PSS trend with managed committed memory. A rising cgroup/PSS/private-dirty line with relatively flat managed memory points toward native/runtime/library/allocator retention, but is not proof that charting owns it. Compare the operation summary and AI intervals before drawing conclusions.

The chart checkpoint measures the immediate before/after operation delta. It does not measure memory retained by the process across operations; use the long-term cgroup/PSS trend for that.